<a href="https://colab.research.google.com/github/Ash100/PD/blob/main/1_Seq_Anal.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#@title ⚙️ Install Required Packages
!pip install biopython pandas matplotlib seaborn requests

from Bio.SeqUtils.ProtParam import ProteinAnalysis
from Bio.Seq import Seq
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
#@title 📥 Upload Your CSV File
import pandas as pd
from google.colab import files

uploaded = files.upload()  # Prompt user to upload CSV
csv_file = next(iter(uploaded))
df = pd.read_csv(csv_file)

# Preview the data
print("Original dataset shape:", df.shape)
df.head()


In [ ]:
#@title ✅ Filter for "Pass" Binders Only
# Normalize the In-silico_filter column in lowercase (in case of inconsistencies)
df['In-silico_filter'] = df['In-silico_filter'].str.strip().str.lower()
df_pass = df[df["In-silico_filter"] == "pass"].copy()

print("Number of PASS entries:", df_pass.shape[0])
df_pass.head()


In [28]:
#@title 🔬 Define Sequence Analysis Function
from Bio.SeqUtils.ProtParam import ProteinAnalysis

def analyze_sequence(seq):
    try:
        seq = seq.replace(' ', '').upper()
        analyzed = ProteinAnalysis(seq)

        mw = analyzed.molecular_weight()
        pI = analyzed.isoelectric_point()
        gravy = analyzed.gravy()
        aromaticity = analyzed.aromaticity()
        instability_index = analyzed.instability_index()
        charge = analyzed.charge_at_pH(7.0)
        length = len(seq)

        # Simple hydrophobic patch score as proxy for aggregation
        agg_score = sum([seq[i] in "VILFYW" for i in range(len(seq))]) / length

        return pd.Series([length, mw, pI, gravy, aromaticity, instability_index, charge, agg_score])
    except:
        return pd.Series([None]*8)


In [29]:
#@title 🧬 Apply Sequence Analysis to Pass Binders
df_pass[['Length', 'MW', 'pI', 'Hydrophobicity', 'Aromaticity',
         'Instability_Index', 'Net_Charge', 'Agg_Patch_Score']] = df_pass['Sequence'].apply(analyze_sequence)

df_pass.head()


,ID,Sequence,In-silico_filter,Binder_pTM,Min_iPAE,Complex_RMSD,Length,MW,pI,Hydrophobicity,Aromaticity,Instability_Index,Net_Charge,Agg_Patch_Score,Aggrescan_Like_Score,Strong_MHC_Binders,MHC_Binder_Class,Binder_Category
3,4,ITGETIANPQNADTAAMFKA,pass,0.36,0.97,1.44,20.0,2064.2761,4.370259,-0.140,0.05,-9.6000,-1.236966,0.15,0,None,Low MHC Binder,High (>0)
22,23,GSATVASQFKAITGKTLPST,pass,0.35,0.92,1.39,20.0,1965.2081,10.002737,0.090,0.05,18.6150,1.758104,0.20,0,None,Low MHC Binder,High (>0)
43,44,DAATASTFYKITGETIVNPG,pass,0.38,0.76,1.57,20.0,2056.2292,4.370259,-0.015,0.10,14.0255,-1.237965,0.25,0,None,Low MHC Binder,High (>0)
50,51,FKKITGKTISNPGNAATAAT,pass,0.37,0.85,1.44,20.0,1991.2487,10.302064,-0.285,0.05,-16.4350,2.757105,0.15,0,None,Low MHC Binder,High (>0)
52,53,SPGNAATAAMFEKITGETVP,pass,0.34,1.00,1.53,20.0,1992.2102,4.529238,-0.035,0.05,24.5950,-1.535232,0.15,0,None,Low MHC Binder,High (>0)


In [30]:
#@title ⚠️ Approximate Aggregation Propensity (AggScore) Based on Hydrophobic Residues

def aggregation_proxy(seq):
    # Simple windowed hydrophobic hot-spot finder
    agg_score = 0
    hydrophobic = "VILFYW"
    for i in range(len(seq)-4):
        window = seq[i:i+5]
        hydros = sum([res in hydrophobic for res in window])
        if hydros >= 4:  # aggregation-prone patch
            agg_score += 1
    return agg_score

df_pass['Aggrescan_Like_Score'] = df_pass['Sequence'].apply(aggregation_proxy)


In [ ]:
#@title 📊 Scientific Visualization of Aggregation Propensity
import matplotlib.pyplot as plt
import seaborn as sns

# Set up seaborn style
sns.set(style="whitegrid", context="talk")

# Color palette
agg_color = "#e74c3c"  # scientific red

# 1. Distribution of aggregation scores
plt.figure(figsize=(10, 6))
sns.histplot(df_pass['Aggrescan_Like_Score'], bins=15, kde=True, color=agg_color, edgecolor='black')
plt.title("Distribution of Aggregation Hotspot Scores", fontsize=16, weight='bold')
plt.xlabel("Aggregation Hotspot Count (Hydrophobic 5-mer Patches)", fontsize=14)
plt.ylabel("Number of Peptides", fontsize=14)
plt.tight_layout()
plt.savefig("agg_score_distribution.png", dpi=600, bbox_inches='tight')
plt.show()

# 2. Top 10 peptides with highest aggregation risk
top_agg = df_pass.sort_values(by='Aggrescan_Like_Score', ascending=False).head(10)

plt.figure(figsize=(12, 6))
sns.barplot(data=top_agg, x='ID', y='Aggrescan_Like_Score', palette="Reds_r", edgecolor='black')
plt.title("Top 10 Peptides with Highest Aggregation Scores", fontsize=16, weight='bold')
plt.xlabel("Peptide ID", fontsize=14)
plt.ylabel("Aggregation Hotspot Count", fontsize=14)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig("top_aggregating_peptides.png", dpi=600, bbox_inches='tight')
plt.show()

# 3. Correlation with Hydrophobicity
plt.figure(figsize=(8, 6))
sns.scatterplot(data=df_pass, x='Hydrophobicity', y='Aggrescan_Like_Score', color="#34495e", s=100, edgecolor='black')
plt.title("Correlation Between Hydrophobicity and Aggregation", fontsize=16, weight='bold')
plt.xlabel("Hydrophobicity (GRAVY Index)", fontsize=14)
plt.ylabel("Aggregation Hotspot Score", fontsize=14)
plt.tight_layout()
plt.savefig("hydrophobicity_vs_agg.png", dpi=600, bbox_inches='tight')
plt.show()


In [38]:
# 1. Identify %Rank columns
rank_cols = [col for col in df_pass.columns if "%Rank" in col]

# 2. Set threshold for strong binder
MHC_STRONG_THRESHOLD = 0.5  # %Rank < 0.5 = strong binder

# 3. Count how many alleles each peptide is a strong binder for
df_pass["Strong_MHC_Binders"] = (df_pass[rank_cols] < MHC_STRONG_THRESHOLD).sum(axis=1)

# 4. Fill NaN with -1 to mark failed predictions (optional)
df_pass["Strong_MHC_Binders"] = df_pass["Strong_MHC_Binders"].fillna(-1).astype(int)


In [39]:
MHC_BINDER_THRESHOLD = 3

df_pass['MHC_Binder_Class'] = df_pass['Strong_MHC_Binders'].apply(
    lambda x: 'Prediction Failed' if x == -1 else (
        'High MHC Binder' if x > MHC_BINDER_THRESHOLD else 'Low MHC Binder'
    )
)


In [40]:
low_mhc = df_pass[df_pass['MHC_Binder_Class'] == 'Low MHC Binder']
high_mhc = df_pass[df_pass['MHC_Binder_Class'] == 'High MHC Binder']
failed_mhc = df_pass[df_pass['MHC_Binder_Class'] == 'Prediction Failed']

print("🔬 Summary of MHC Binding Classification")
print(f"Total peptides analyzed: {len(df_pass)}")
print(f"✅ Valid predictions: {len(low_mhc) + len(high_mhc)}")
print(f"   - Low MHC Binders (≤ {MHC_BINDER_THRESHOLD}): {len(low_mhc)}")
print(f"   - High MHC Binders (> {MHC_BINDER_THRESHOLD}): {len(high_mhc)}")
print(f"⚠️ Failed predictions: {len(failed_mhc)}\n")

# Optional: Show some entries
print("🟢 Low MHC Binders:")
print(low_mhc[['ID', 'Strong_MHC_Binders']].head())

print("\n🔥 High MHC Binders:")
print(high_mhc[['ID', 'Strong_MHC_Binders']].head())

print("\n⚠️ Failed MHC Predictions:")
print(failed_mhc[['ID', 'Strong_MHC_Binders']])


🔬 Summary of MHC Binding Classification
Total peptides analyzed: 27
✅ Valid predictions: 27
   - Low MHC Binders (≤ 3): 27
   - High MHC Binders (> 3): 0
⚠️ Failed predictions: 0

🟢 Low MHC Binders:
    ID  Strong_MHC_Binders
3    4                   0
22  23                   0
43  44                   0
50  51                   0
52  53                   0

🔥 High MHC Binders:
Empty DataFrame
Columns: [ID, Strong_MHC_Binders]
Index: []

⚠️ Failed MHC Predictions:
Empty DataFrame
Columns: [ID, Strong_MHC_Binders]
Index: []


In [ ]:
#@title 🎨 Publication-Grade Visualization for Sequence-Based Properties (with Aggregation + Immunogenicity)

import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl

# Set general seaborn + matplotlib style
sns.set(style="whitegrid", context="talk")
mpl.rcParams.update({
    "axes.titlesize": 16,
    "axes.labelsize": 14,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "legend.fontsize": 12,
    "axes.linewidth": 1.5,
    "axes.edgecolor": "black",
    "figure.dpi": 300,
    "savefig.dpi": 600,
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial"],
})

# Define color palette
palette = sns.color_palette("Set2")

# Define full list of properties to plot (include new ones)
properties = [
    ("MW", "Molecular Weight (Da)"),
    ("pI", "Isoelectric Point (pI)"),
    ("Hydrophobicity", "Hydrophobicity (GRAVY Index)"),
    ("Instability_Index", "Instability Index"),
    ("Net_Charge", "Net Charge (pH 7.0)"),
    ("Agg_Patch_Score", "Hydrophobic Patch Ratio"),
    ("Aggrescan_Like_Score", "Aggregation Hotspot Score"),
    ("Strong_MHC_Binders", "Number of Strong MHC Binders"),
]

# Create subplots (4x2 grid)
fig, axes = plt.subplots(4, 2, figsize=(16, 16))
axes = axes.flatten()

# Plot each property with KDE histograms
for i, (col, label) in enumerate(properties):
    if col in df_pass.columns:
        sns.histplot(df_pass[col].dropna(), ax=axes[i], kde=True, color=palette[i % len(palette)], edgecolor='black')
        axes[i].set_title(f"Distribution of {label}", fontsize=16, weight='bold')
        axes[i].set_xlabel(label, fontsize=14)
        axes[i].set_ylabel("Count", fontsize=14)
        axes[i].tick_params(width=1.5)
    else:
        axes[i].axis('off')  # Hide plot if column not found

# Layout and export
plt.tight_layout()
plt.savefig("sequence_property_distributions_full.png", dpi=600, bbox_inches='tight')
pl


## 🧪 Scientific Explanation of Sequence-Based Properties

This section summarizes all the calculated descriptors for macrocyclic peptide binders against the PagC receptor. These properties were used to evaluate biophysical stability, solubility, aggregation risk, and immunogenic potential prior to molecular dynamics simulations and experimental validation.

---

### 1. **Molecular Weight (MW)**
- **Definition:** The total mass of the peptide in Daltons.
- **Relevance:** Affects peptide diffusion, cell permeability, and synthesis cost. Ideal binders usually fall in a target range to ensure drug-like behavior.

---

### 2. **Isoelectric Point (pI)**
- **Definition:** The pH at which the peptide has zero net charge.
- **Relevance:** Useful for predicting peptide solubility, interaction with target surfaces, and behavior in purification buffers. Extreme pI values may reduce stability or solubility.

---

### 3. **Hydrophobicity (GRAVY Index)**
- **Definition:** Grand Average of Hydropathy; the mean hydropathy score across all residues.
- **Relevance:** Balances peptide solubility and target binding. High hydrophobicity can indicate poor solubility or potential aggregation, but may improve membrane interaction.

---

### 4. **Instability Index**
- **Definition:** A computed score estimating peptide stability in vitro.
- **Relevance:** Peptides with an index >40 are considered unstable and prone to degradation. Stable binders are preferred for therapeutic development.

---

### 5. **Net Charge at pH 7.0**
- **Definition:** Net electrostatic charge of the peptide under physiological conditions.
- **Relevance:** Influences solubility, diffusion, and binding. Neutral to moderately charged peptides are often optimal for receptor interactions and reduced immunogenicity.

---

### 6. **Hydrophobic Patch Ratio (Agg_Patch_Score)**
- **Definition:** Proportion of hydrophobic 5-residue windows across the peptide sequence.
- **Relevance:** A proxy for aggregation-prone motifs, modeled after tools like TANGO and Aggrescan. High values suggest greater aggregation potential.

---

### 7. **Aggregation Hotspot Score (Aggrescan_Like_Score)**
- **Definition:** A sequence-derived score simulating aggregation-prone regions based on consecutive hydrophobic and β-sheet-favoring residues.
- **Relevance:** Indicates likelihood of spontaneous aggregation or self-assembly. Peptides with lower scores are preferred for therapeutic and formulation stability.

---

### 8. **Number of Strong MHC Binders (Strong_MHC_Binders)**
- **Definition:** Count of predicted strong HLA class I or II binders from the peptide, using NetMHCpan or IEDB tools.
- **Relevance:** Measures potential immunogenicity. Peptides with high MHC affinity may elicit T-cell responses and are less suitable for therapeutic use unless immune activation is desired.

---

These descriptors enable a rational, multi-parametric ranking of candidate peptides before high-cost simulations or wet-lab testing. All metrics can be used for filtering, scoring, and visualization in downstream analyses.
